# Aggregate viruses → monthly detections

Consolidate `viruses_normalized.parquet` to H3 × month × pathogen × host rows (spec `04`).

| Item | Detail |
|------|--------|
| Input | `data/02_interim/viruses_normalized.parquet` |
| Output | `data/03_processed/virus_monthly.parquet` |
| Grain | `(h3_cell, country_iso2, year, month, pathogen_name, host_group, host)` |
| Measures | `detections_count` (row count) |

Inclusion: non-empty `h3_cell` and non-null `year` / `month`. Other group keys may be empty. Additive analysis table — does not feed merge notebook `10`.

Resolve repo paths and load the interim virus table.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import default_interim_dir, default_processed_dir, find_repo_root
from vector_guard.resolve import load_interim_viruses
from vector_guard.virus_agg import (
    VIRUS_MONTHLY_COLS,
    VIRUS_MONTHLY_KEYS,
    aggregate_virus_monthly,
)

REPO_ROOT = find_repo_root(Path.cwd())
INTERIM_DIR = default_interim_dir(REPO_ROOT)
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = PROCESSED_DIR / "virus_monthly.parquet"

viruses = load_interim_viruses(INTERIM_DIR)
(REPO_ROOT, OUT_PATH, len(viruses))

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/03_processed/virus_monthly.parquet'),
 168307)

Aggregate to virus-monthly grain, print key counts, write the parquet.

In [2]:
n_in = len(viruses)
key_ok = pd.Series(True, index=viruses.index)
for col in ("h3_cell", "year", "month"):
    s = viruses[col]
    if pd.api.types.is_numeric_dtype(s):
        key_ok &= s.notna()
    else:
        as_str = s.astype("string")
        key_ok &= as_str.notna() & (as_str.str.strip() != "")
n_kept = int(key_ok.sum())
n_dropped = n_in - n_kept

agg = aggregate_virus_monthly(viruses)
assert list(agg.columns) == VIRUS_MONTHLY_COLS
assert set(VIRUS_MONTHLY_KEYS).issubset(agg.columns)

print(f"Input rows: {n_in}")
print(f"Rows with h3_cell + year + month: {n_kept}")
print(f"Rows dropped (missing h3/year/month): {n_dropped}")
print(f"Output rows: {len(agg)}")
print(f"detections_count sum: {int(agg['detections_count'].sum())}")
assert int(agg["detections_count"].sum()) == n_kept

display(agg.head(10))
display(agg.describe(include="all"))

agg.to_parquet(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} ({len(agg)} rows)")

Input rows: 168307
Rows with h3_cell + year + month: 64283
Rows dropped (missing h3/year/month): 104024
Output rows: 17814
detections_count sum: 64283


,h3_cell,country_iso2,year,month,pathogen_name,host_group,host,detections_count
0,8408831ffffffff,FI,2012,10,Schmallenberg virus,livestock,Domestic - cattle,1
1,84089a3ffffffff,EE,2024,10,West Nile virus,wildlife_mammal,Wild - Goshawk,1
2,84089a7ffffffff,EE,2013,1,Schmallenberg virus,livestock,Domestic - sheep,1
3,8408b3dffffffff,SE,1990,1,Dengue virus,human,Homo sapiens,6
4,8408b3dffffffff,SE,1990,2,Dengue virus,human,Homo sapiens,9
5,8408b47ffffffff,SE,2024,9,Bluetongue virus,livestock,Domestic - Cattle | Domestic - Sheep,1
6,8408b49ffffffff,SE,2024,9,Bluetongue virus,livestock,Domestic - Cattle,1
7,8408b4dffffffff,SE,2024,9,Bluetongue virus,livestock,Domestic - Sheep | Domestic - Cattle,1
8,8408f5dffffffff,SE,2013,9,Sindbis virus,mosquito,Culiseta morsitans,1
9,8408f5dffffffff,SE,2015,6,Inkoo orthobunyavirus,mosquito,Ochlerotatus communis,1


,h3_cell,country_iso2,year,month,pathogen_name,host_group,host,detections_count
count,17814,17814,17814.0,17814.0,17814,17814,17814,17814.000000
unique,3510,154,<NA>,<NA>,25,10,786,NaN
top,8429a87ffffffff,US,<NA>,<NA>,Bluetongue virus,livestock,Homo sapiens,NaN
freq,368,2004,<NA>,<NA>,5930,9428,3532,NaN
mean,NaN,NaN,2016.742337,7.214831,NaN,NaN,NaN,3.608566
std,NaN,NaN,6.819152,3.156546,NaN,NaN,NaN,8.421353
min,NaN,NaN,1931.0,1.0,NaN,NaN,NaN,1.000000
25%,NaN,NaN,2012.0,5.0,NaN,NaN,NaN,1.000000
50%,NaN,NaN,2017.0,8.0,NaN,NaN,NaN,1.000000
75%,NaN,NaN,2023.0,10.0,NaN,NaN,NaN,3.000000


Wrote /workspaces/vector-guard/data/03_processed/virus_monthly.parquet (17814 rows)
